In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from tokenizers.processors import TemplateProcessing
from pathlib import Path

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/SLM/Data")
train_path = "/content/drive/MyDrive/SLM/Data/train.txt"
val_path = "/content/drive/MyDrive/SLM/Data/val.txt"

In [ ]:
vocab_size = 8000

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [ ]:
trainer = trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]", "<user>", "<assistant>", "<eos>"],
)

In [ ]:
files = [str(train_path), str(val_path)]
tokenizer.train(files, trainer)

In [ ]:
tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    pair="[BOS] $A [EOS] $B:1 [EOS]:1",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]")),
    ],
)

In [ ]:
tokenizer.save(str(DATA_DIR / "tokenizer.json"))
print("Saved tokenizer with vocab size:", tokenizer.get_vocab_size())

In [ ]:
import torch

In [ ]:
def encode_file(path, tokenizer):
    text = Path(path).read_text(encoding="utf-8")
    ids = tokenizer.encode(text).ids
    return torch.tensor(ids, dtype=torch.long)

In [ ]:
tokenizer = Tokenizer.from_file(str(DATA_DIR / "tokenizer.json"))

In [ ]:
def encode_file_streaming(txt_path, out_pt_path, chunk_size=1_000_000):
    """
    Reads the text file in chunks, tokenizes incrementally,
    and writes a single large tensor safely.
    """
    all_ids = []
    with open(txt_path, "r", encoding="utf-8") as f:
        buffer = ""
        for line in f:
            buffer += line
            if len(buffer) >= chunk_size:
                ids = tokenizer.encode(buffer).ids
                all_ids.extend(ids)
                buffer = ""

        # encode leftover buffer
        if buffer:
            ids = tokenizer.encode(buffer).ids
            all_ids.extend(ids)

    ids_tensor = torch.tensor(all_ids, dtype=torch.long)
    torch.save(ids_tensor, out_pt_path)
    print("Saved:", out_pt_path, "Total tokens:", ids_tensor.numel())


In [ ]:
encode_file_streaming(train_path, DATA_DIR / "train_ids.pt")
encode_file_streaming(val_path, DATA_DIR / "val_ids.pt")

Saved: /content/drive/MyDrive/SLM/Data/train_ids.pt Total tokens: 175510419
Saved: /content/drive/MyDrive/SLM/Data/val_ids.pt Total tokens: 1771853
